In [5]:
#forces us to reload the libraries each time we do a function call,
#so I can update things in real time
%load_ext autoreload
%autoreload 2
#import path libraries
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from utils import psf
from utils import plotting
from utils import zernike
from optimization import reconstruction
from tqdm import tqdm
from systematic_testing import correction


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
plt.style.use("bmh")
plt.rcParams["figure.figsize"] = [548 / 72, 390 / 72]
plt.rcParams["font.size"] = 12
plt.rcParams['text.usetex'] = True
plt.rcParams["figure.dpi"] = 300

In [7]:
N_order = 3                # 3 for 3P, 2 for 2P
lambd = 1.3e-3             # Wavelength [mm]
n = 1.333                  # Refractive index
k = 2*n*np.pi/lambd        # Wavenumber
num_apt = 1.05             # Numerical aperture
f = 7.2                    # Focal length of objective (Olympus) [mm]
grid_size = 128            # BFP resolution
L_bfp = 15.12              # BFP diameter [mm]
mag = 4                    # Magnification rate from input to objective
w_0 = 3.5                  # mm
num_apt = 1.05
L_ffp = 0.0065             # FFP size [mm]
grid_ffp = 64
grid_bfp = 64

Compare repeated stack fitting with a modal Booth $2N+1$ baseline. The sample and unknown wavefront remain fixed within each trial; the SLM phase accumulates, and each acquisition observes the remaining aberration plus its temporary probe.

Exposure is calibrated once from the noiseless initial zero-bias image at z=0, then shared by both methods. `INITIAL_REFERENCE_SNR` sets this reference exposure; subsequent SNR changes naturally with brightness. Every measured z-plane counts as one image. RMS is a simulation diagnostic and does not guide either correction algorithm. The SLM is ideal, with no drift, bleaching, or calibration error.

The Booth baseline uses mean fluorescence intensity and bounded three-point quadratic fits, with one baseline shared across all modes before applying their updates together. This is the basic modal benchmark, without orthogonalization for the image metric. [Method reference](https://doi.org/10.1364/OE.20.002598).


In [8]:
microscope = psf.Microscope(N_order = N_order,
                            lambd = lambd,
                            n = n,
                            num_apt = num_apt,
                            f = f,
                            mag = mag,
                            w_0 = w_0,
                            L_bfp = L_bfp,
                            grid_bfp = grid_bfp)

grid = psf.Centered_Square_Grid(L_ffp = L_ffp,
                                grid_ffp = grid_ffp,
                                z_level = 0)

xs, ys = grid.get_xy()


modes = [[-2,2],
         [2,2],
         [-3,3],
         [-1,3],
         [1,3],
         [3,3],
         [-4,4],
         [-2,4],
         [0,4],
         [2,4],
         [4,4],
         [-5,5],
         [-3, 5],
         [-1, 5],
         [1, 5],
         [3, 5],
         [5,5]]


In [9]:
NUM_TRIALS = 1
ABERRATION_RMS = 0.15
ABERRATION_SEED = 0
NOISE_SEED = 1
INITIAL_REFERENCE_SNR = 50.0
RHO = 1e46
IMAGE_BUDGET = 310
OPTICAL_MODE = "vector"
CORRECTION_MODES = [mode.copy() for mode in modes]
CORRECTION_GAIN = 1.0
MAX_MODAL_UPDATE = 0.10  # Maximum SLM increment per mode per update, in waves.

NUM_STACK_Z_PLANES = 5
STACK_FOCAL_POSITIONS = np.linspace(-0.002, 0.002, NUM_STACK_Z_PLANES)
STACK_DIVERSITY_MODE = [0, 4]
STACK_DIVERSITY_STRENGTHS = np.array([-0.10, 0.0, 0.10])
OPTIMIZER_OPTIONS = dict(method="newton", max_iterations=20, verbose=False)

BOOTH_FOCAL_POSITIONS = np.array([0.0])
BOOTH_BIAS = 0.10  # Probe coefficient in waves; adjust independently of the stack diversities.
BOOTH_METRIC = correction.mean_brightness

z_sample_levels = np.linspace(-0.002, 0.002, 41)
mask_3D = psf.bead_img_3D(
    grid=grid, z_levels=z_sample_levels,
    xs=[0.0], ys=[0.0], zs=[0.0], bead_sizes=[0.001],
)
stack_images_per_update = len(STACK_FOCAL_POSITIONS) * len(STACK_DIVERSITY_STRENGTHS)
booth_images_per_update = len(BOOTH_FOCAL_POSITIONS) * (2 * len(CORRECTION_MODES) + 1)


In [10]:
correction_truths = []
correction_exposures = []
stack_histories = []
booth_histories = []
correction_total_updates = NUM_TRIALS * (
    IMAGE_BUDGET // stack_images_per_update + IMAGE_BUDGET // booth_images_per_update
)

with tqdm(total=correction_total_updates, desc="Correction comparison") as progress:
    for trial in range(NUM_TRIALS):
        pattern_rng = np.random.default_rng(np.random.SeedSequence([ABERRATION_SEED, trial]))
        truth = zernike.RMSAberration(
            modes=modes, raw_strengths=pattern_rng.normal(0, 1, len(modes)),
            RMS_desired=ABERRATION_RMS, alpha=microscope.alpha,
        )
        correction_truths.append(truth)
        # Simulation calibration, not an acquired measurement used by either method.
        _, _, _, reference_images = microscope.compute_image_3D(
            image=mask_3D, aberration=truth,
            focal_z_levels=np.array([0.0]), mode=OPTICAL_MODE,
        )
        exposure = correction.exposure_for_snr(reference_images, INITIAL_REFERENCE_SNR)
        correction_exposures.append(exposure)

        def make_simulator(method_index):
            return correction.SimulatedSLM(
                microscope=microscope, sample=mask_3D, truth=truth,
                correction_modes=CORRECTION_MODES, exposure=exposure,
                rng=np.random.default_rng(np.random.SeedSequence([NOISE_SEED, trial, method_index])),
                optical_mode=OPTICAL_MODE,
            )

        stack_histories.append(correction.stack_feedback(
            simulator=make_simulator(0), grid=grid,
            sample_z_levels=z_sample_levels,
            focal_positions=STACK_FOCAL_POSITIONS,
            diversity_mode=STACK_DIVERSITY_MODE,
            diversity_strengths=STACK_DIVERSITY_STRENGTHS,
            rho=RHO, image_budget=IMAGE_BUDGET,
            gain=CORRECTION_GAIN, max_update=MAX_MODAL_UPDATE,
            optimizer_kwargs=OPTIMIZER_OPTIONS,
            on_update=lambda _: progress.update(1),
        ))
        booth_histories.append(correction.booth_feedback(
            simulator=make_simulator(1), focal_positions=BOOTH_FOCAL_POSITIONS,
            bias=BOOTH_BIAS, image_budget=IMAGE_BUDGET,
            gain=CORRECTION_GAIN, max_update=MAX_MODAL_UPDATE,
            metric=BOOTH_METRIC,
            on_update=lambda _: progress.update(1),
        ))

# Keep the previous committed correction between complete acquisitions/sweeps.
correction_image_grid = np.arange(IMAGE_BUDGET + 1)
stack_rms_by_images = np.array([
    correction.rms_at_image_counts(history, correction_image_grid)
    for history in stack_histories
])
booth_rms_by_images = np.array([
    correction.rms_at_image_counts(history, correction_image_grid)
    for history in booth_histories
])


Correction comparison:   0%|          | 0/28 [00:13<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(dpi=150)
for label, rms_values in [
    ("Stack feedback", stack_rms_by_images),
    (r"Booth $2N+1$", booth_rms_by_images),
]:
    mean_rms = rms_values.mean(axis=0)
    std_rms = rms_values.std(axis=0)
    line, = ax.step(correction_image_grid, mean_rms, where="post", label=label)
    ax.fill_between(
        correction_image_grid, np.maximum(0, mean_rms - std_rms), mean_rms + std_rms,
        step="post", color=line.get_color(), alpha=0.2,
    )
ax.set_xlabel("Images taken")
ax.set_ylabel("Residual wavefront RMS [waves]")
ax.set_xlim(0, IMAGE_BUDGET)
ax.set_ylim(bottom=0)
ax.legend()
fig.savefig("correction_rms_vs_images.png", bbox_inches="tight")
plt.show()


In [ ]:
# Numeric results reload without pickled objects; detailed histories remain in memory.
np.savez_compressed(
    "correction_results.npz",
    images_taken=correction_image_grid,
    stack_rms=stack_rms_by_images,
    booth_rms=booth_rms_by_images,
    truth_modes=np.asarray(modes),
    correction_modes=np.asarray(CORRECTION_MODES),
    truth_strengths=np.array([truth.strengths for truth in correction_truths]),
    exposures=np.asarray(correction_exposures),
    stack_update_images=np.array([[p["images_taken"] for p in h] for h in stack_histories]),
    booth_update_images=np.array([[p["images_taken"] for p in h] for h in booth_histories]),
    stack_slm_strengths=np.array([[p["slm_strengths"] for p in h] for h in stack_histories]),
    booth_slm_strengths=np.array([[p["slm_strengths"] for p in h] for h in booth_histories]),
    stack_detected_photons=np.array([[p["photons_detected"] for p in h] for h in stack_histories]),
    booth_detected_photons=np.array([[p["photons_detected"] for p in h] for h in booth_histories]),
    rho=RHO, initial_reference_snr=INITIAL_REFERENCE_SNR,
    aberration_rms=ABERRATION_RMS, image_budget=IMAGE_BUDGET,
    stack_focal_positions=STACK_FOCAL_POSITIONS,
    stack_diversity_mode=STACK_DIVERSITY_MODE,
    stack_diversity_strengths=STACK_DIVERSITY_STRENGTHS,
    booth_focal_positions=BOOTH_FOCAL_POSITIONS, booth_bias=BOOTH_BIAS,
    correction_gain=CORRECTION_GAIN, max_modal_update=MAX_MODAL_UPDATE,
)
